In [ ]:
# Uncomment in a fresh Kaggle notebook environment.
%pip install -q unsloth datasets trl transformers==4.56.2 accelerate peft bitsandbytes pandas lxml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 148.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.2/403.2 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 131.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from google.colab import drive

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Torch: 2.10.0+cu128
CUDA available: True


In [ ]:
# Core training config.
CONFIG = {
    "model_name": "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit",
    "max_seq_length": 4096,
    "lora_r": 32,
    "lora_alpha": 64,
    "learning_rate": 2e-4,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 32,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "logging_steps": 5,
    "eval_steps": 5,
    "save_steps": 200,
    "max_train_samples_per_source": 50000,
    "eval_size": 0.02,
    "output_dir": "/finetunedmodel",
    "eval_strategy": "steps",
    "logging_strategy": "steps",
}

SYSTEM_PROMPT = (
    "You are an SVG code generator. Given a description, output only valid SVG code, nothing else. "
    "Only use these elements: svg, g, path, rect, circle, ellipse, line, polyline, polygon, "
    "defs, use, symbol, clipPath, mask, linearGradient, radialGradient, stop, text, tspan, title, "
    "desc, style, pattern, marker, filter."
    "Keep the final SVG code strictly under 2048 tokens"
)

CONFIG

{'model_name': 'unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit',
 'max_seq_length': 4096,
 'lora_r': 32,
 'lora_alpha': 64,
 'learning_rate': 0.0002,
 'num_train_epochs': 1,
 'per_device_train_batch_size': 2,
 'gradient_accumulation_steps': 32,
 'warmup_ratio': 0.05,
 'weight_decay': 0.01,
 'logging_steps': 5,
 'eval_steps': 5,
 'save_steps': 200,
 'max_train_samples_per_source': 50000,
 'eval_size': 0.02,
 'output_dir': '/finetunedmodel',
 'eval_strategy': 'steps',
 'logging_strategy': 'steps'}

In [ ]:
#Mound to drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/DL-midterm-2026')

#Perform data loading and manipulation from general experiment 1
df = pd.read_csv("train.csv")
df.drop('id', axis = 1)
df = df[['prompt', 'svg']].astype('string')
df['svg'] = df['svg'].str.replace(r'\d+\.\d+', lambda m: f"{float(m.group()):.1f}".rstrip('0').rstrip('.'), regex = True)
df = df[df['svg'].str.contains(r'viewBox="0 0 200 200"', regex = True, na = False)]
lengths = df['svg'].str.len()
threshold = lengths.quantile(0.998)
df = df[lengths <= threshold]
#Reduce dataset to 2000 rows for quick learning rate experimentation
df = df.sample(n = 2000)

Mounted at /content/drive


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

def build_conversations(row):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["prompt"]},
        {"role": "assistant", "content": row["svg"]}
    ]
    formatted_string = tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt = False)
    return formatted_string

df['text'] = df.apply(build_conversations, axis = 1)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

In [ ]:
hf_dataset = Dataset.from_pandas(df)

split_dataset = hf_dataset.train_test_split(test_size = CONFIG['eval_size'], seed = SEED)
train_ds = split_dataset['train']
eval_ds = split_dataset['test']

print(f"Train rows: {len(train_ds)}")
print(f"Eval rows: {len(eval_ds)}")
print(train_ds[:1])

Train rows: 1960
Eval rows: 40
{'prompt': ['A simple gear icon with a circular center and toothed outline.'], 'svg': ['<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 200 200" height="200px" width="200px"><path fill="#333333" fill-opacity="1"  filling="0" d="M188.6 79.3 A35.2 35.2 0 0 1 165.8 62.6 A34.1 34.1 0 0 1 162.6 34.9 A4.3 4.3 0 0 0 161.3 30.3 A93.9 93.9 0 0 0 131.3 12.8 A4.5 4.5 0 0 0 126.5 14 A35.6 35.6 0 0 1 100 25.3 A35.6 35.6 0 0 1 74 14.2 A4.5 4.5 0 0 0 68.8 12.8 A93.5 93.5 0 0 0 38.8 30.4 A4.3 4.3 0 0 0 37.4 35 A34.1 34.1 0 0 1 34.3 62.6 A35.2 35.2 0 0 1 11.4 79.3 A4.4 4.4 0 0 0 8 82.8 A88.9 88.9 0 0 0 8 117.2 A4.4 4.4 0 0 0 11.4 120.7 A35.2 35.2 0 0 1 34.2 137.3 C39.2 145.6 40.3 155.7 37.4 165 A4.3 4.3 0 0 0 38.8 169.6 A93.9 93.9 0 0 0 68.8 187 L70.2 187 A4.5 4.5 0 0 0 73.5 185.7 A36 36 0 0 1 125.6 185.7 C126.9 187.4 129.3 188 131.3 187 A93.6 93.6 0 0 0 161.8 169.7 A4.3 4.3 0 0 0 163.1 165.1 A34.1 34.1 0 0 1 165.8 137.3 A35.2 35.2 0 0 1 188.6 120.7 A4.4 4.4 0 0 0 19

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

Unsloth 2026.3.17 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
import warnings

# Suppress specific Hugging Face FutureWarnings
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="transformers.*"
)

learningRates = [1e-6, 2e-6, 5e-6, 1e-5, 2e-5, 5e-5, 1e-4, 2e-4, 5e-6]

for lr in learningRates:

    training_args = TrainingArguments(
        output_dir=CONFIG["output_dir"],
        num_train_epochs=CONFIG["num_train_epochs"],
        per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
        gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
        learning_rate=lr,
        warmup_ratio=CONFIG["warmup_ratio"],
        weight_decay=CONFIG["weight_decay"],
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=CONFIG["logging_steps"],
        eval_strategy="steps",
        eval_steps=CONFIG["eval_steps"],
        save_steps=CONFIG["save_steps"],
        save_total_limit=2,
        report_to="none",
        optim="paged_adamw_8bit",
        lr_scheduler_type="cosine",
        seed=SEED,
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        dataset_text_field="text",
        max_seq_length=CONFIG["max_seq_length"],
        packing=True,
        args=training_args,
    )
    print(f"Experiement with learning rate: {lr}")
    train_result = trainer.train()
    train_result

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/1960 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/40 [00:00<?, ? examples/s]

Experiement with learning rate: 1e-06


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,960 | Num Epochs = 1 | Total steps = 31
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 32
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 32 x 1) = 64
 "-____-"     Trainable parameters = 59,867,136 of 3,145,805,824 (1.90% trained)


Step,Training Loss,Validation Loss
5,1.041700,1.066701
10,1.015400,1.065644
15,1.041800,1.064438
20,1.040900,1.064123
25,1.044400,1.063766
30,1.052800,1.063938


...

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/40 [00:00<?, ? examples/s]

Experiement with learning rate: 5e-06


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,960 | Num Epochs = 1 | Total steps = 31
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 32
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 32 x 1) = 64
 "-____-"     Trainable parameters = 59,867,136 of 3,145,805,824 (1.90% trained)


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [1]:
"""
I ran this cell because 5e-04 wasn't tested earlier due to a typo. I accidentally overid the output, but it was

LR: 5e-04

Step Training Loss Validation Loss

5 0.428900 0.513873

10 0.443800 0.473333

15 0.438600 0.459409

20 0.438800 0.453902

25 0.434100 0.451613

30 0.438800 0.450449
"""

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=5e-4,
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=True,
    args=training_args,
)
print(f"Experiement with learning rate: {5e-4}")
train_result = trainer.train()
train_result

NameError: name 'TrainingArguments' is not defined

Learning rate of 2e-04 demonstrates the best tradeoff bias and variance.